In [ ]:
import os
import glob
import pandas as pd

# 기준 폴더
base_path = r"C:\startcoding\GUKTO\아파트_매매"

# 통합 폴더
merge_folder = os.path.join(base_path, "아파트_매매_통합")
os.makedirs(merge_folder, exist_ok=True)

# 시도 폴더 목록
folders = [
    f for f in os.listdir(base_path)
    if os.path.isdir(os.path.join(base_path, f))
    and f != "아파트_매매_통합"
]

for region in folders:

    region_path = os.path.join(base_path, region)

    excel_files = glob.glob(os.path.join(region_path, "*.xlsx"))

    merged_df = pd.DataFrame()

    for file in excel_files:

        try:

            df = pd.read_excel(
                file,
                skiprows=13,
                header=None
            ).iloc[:, :21]

            merged_df = pd.concat([merged_df, df], ignore_index=True)

            print(f"{region} 추가: {os.path.basename(file)}")

        except Exception as e:

            print(f"오류: {file} / {e}")

    # 시도별 저장
    output_path = os.path.join(
        merge_folder,
        f"{region}_아파트_통합.xlsx"
    )

    merged_df.to_excel(output_path, index=False, header=False)

    print(f"{region} 통합 완료")
    print(output_path)

In [14]:
import pandas as pd
import numpy as np
import os

input_folder = r"C:\startcoding\GUKTO\아파트_매매\아파트_매매_통합\정제할 통합파일"
output_folder = r"C:\startcoding\GUKTO\아파트_매매\아파트_매매_정제"

os.makedirs(output_folder, exist_ok=True)

files = [f for f in os.listdir(input_folder) if f.endswith(".xlsx")]

for file in files:
    print(f"처리중: {file}")
    path = os.path.join(input_folder, file)
    df = pd.read_excel(path, header=None)

    df_clean = pd.DataFrame()

    # 1. 컬럼 매핑 (두 번째 이미지 기준 인덱스)
    df_clean['시군구'] = df.iloc[:, 1]
    df_clean['번지'] = df.iloc[:, 2]
    df_clean['본번'] = df.iloc[:, 3]
    df_clean['부번'] = df.iloc[:, 4]
    df_clean['단지명'] = df.iloc[:, 5]
    df_clean['전용면적'] = df.iloc[:, 6]
    df_clean['계약년월'] = df.iloc[:, 7]
    df_clean['계약일'] = df.iloc[:, 8]
    df_clean['거래금액'] = df.iloc[:, 9]
    df_clean['층'] = df.iloc[:, 11]
    df_clean['매수자'] = df.iloc[:, 12]
    df_clean['매도자'] = df.iloc[:, 13]
    df_clean['건축년도'] = df.iloc[:, 14]
    df_clean['도로명'] = df.iloc[:, 15]

    # 2. 해제거래 제거 (16번 인덱스 기준)
    if df.shape[1] > 16:
        cancel_val = df.iloc[:, 16].astype(str).str.strip()
        df_clean = df_clean[~cancel_val.str.match(r'^\d{4,8}$')]

    # 3. 데이터 타입 변환 (거래금액 콤마 제거 및 숫자화)
    df_clean['거래금액'] = df_clean['거래금액'].astype(str).str.replace(',', '')
    num_cols = ['전용면적', '계약년월', '계약일', '거래금액', '층', '건축년도']
    for col in num_cols:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

    # 4. 날짜 및 연식 계산 (계산용 임시 날짜 컬럼 생성)
    temp_dt = pd.to_datetime(
        df_clean['계약년월'].astype(str) + 
        df_clean['계약일'].astype(int).astype(str).str.zfill(2), 
        format='%Y%m%d', errors='coerce'
    )
    
    # [수정] 계약일자를 6자리 YYMMDD 문자로 변환
    df_clean['계약일자'] = temp_dt.dt.strftime('%y%m%d')
    # [수정] 연식 계산은 임시 날짜(temp_dt) 활용
    df_clean['연식'] = temp_dt.dt.year - df_clean['건축년도']

    # 5. 주소 분해 (최대 5개)
    addr_split = df_clean['시군구'].astype(str).str.split(' ', expand=True)
    for i in range(5):
        df_clean[f'주소{i+1}'] = addr_split[i] if i < addr_split.shape[1] else ""

    # 6. 파생 변수 (단가 정수화 및 규모 계산)
    df_clean['면적규모'] = (df_clean['전용면적'] / 30).round() * 30
    df_clean['연식규모'] = (df_clean['연식'] / 10).round() * 10
    # [수정] 단가는 정수로 (결측치는 0 처리)
    df_clean['단가'] = (df_clean['거래금액'] / df_clean['전용면적']).replace([np.inf, -np.inf], np.nan).fillna(0).astype(int)

    # 7. 순서번호 및 최종 정리
    df_clean = df_clean.reset_index(drop=True)
    df_clean['순서번호'] = df_clean.index + 1

    final_cols = [
        '순서번호', '주소1', '주소2', '주소3', '주소4', '주소5',
        '번지', '도로명', '단지명', '매수자', '매도자',
        '계약일자', '거래금액', '면적규모', '연식규모',
        '전용면적', '연식', '층', '단가'
    ]

    df_final = df_clean[final_cols]

    # 8. 저장
    save_path = os.path.join(output_folder, file.replace("통합", "정제"))
    df_final.to_excel(save_path, index=False)
    print(f"완료: {save_path}")

print("전체 작업 완료!")

처리중: 경기도_아파트_통합.xlsx
완료: C:\startcoding\GUKTO\아파트_매매\아파트_매매_정제\경기도_아파트_정제.xlsx
처리중: 대전광역시_아파트_통합.xlsx
완료: C:\startcoding\GUKTO\아파트_매매\아파트_매매_정제\대전광역시_아파트_정제.xlsx
처리중: 세종특별자치시_아파트_통합.xlsx
완료: C:\startcoding\GUKTO\아파트_매매\아파트_매매_정제\세종특별자치시_아파트_정제.xlsx
처리중: 충청남도_아파트_통합.xlsx
완료: C:\startcoding\GUKTO\아파트_매매\아파트_매매_정제\충청남도_아파트_정제.xlsx
전체 작업 완료!
